In [14]:

def parse_rules_to_cluster_df(file_path: str) -> pd.DataFrame:
    """
    Parse a YARA rules file, extract detection/total values, and aggregate by cluster.
    
    Args:
        file_path (str): Path to the YARA rules file.
    
    Returns:
        pd.DataFrame: DataFrame with index = Cluster names and columns = Detection (avg), Total.
    """
    try:
        with open(file_path, 'r') as f:
            file_content = f.read()
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading file: {e}")
        return pd.DataFrame()

    # Match rules: ClusterName_run  Detection  Total
    rule_pattern = r'rule\s+(\w+)\s*{[^}]*//Input TP Rate:\s*//(\d+)/(\d+)[^}]*}'
    matches = re.findall(rule_pattern, file_content, re.MULTILINE)

    if not matches:
        print("No rules with Input TP Rate found in the file.")
        return pd.DataFrame()

    # Build initial DataFrame
    df = pd.DataFrame(
        [(name, int(det), int(total)) for name, det, total in matches],
        columns=["RuleName", "Detection", "Total"]
    )

    # Extract cluster name (before "_")
    df["Cluster"] = df["RuleName"].str.split("_").str[0]

    # Aggregate: average detection, take the first total (they are same across runs)
    cluster_df = df.groupby("Cluster").agg({
        "Detection": "mean",
        "Total": "first"
    })

    return cluster_df

In [33]:
a=0
t=0
file_path =[ "/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th50rules/merged_group_1.yar",
            "/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th60rules/merged_group_1.yar",
            "/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th70rules/merged_group_1.yar",
            "/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th80rules/merged_group_1.yar",
            "/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th90rules/merged_group_1.yar"]
for path in file_path:             
    df = parse_rules_to_cluster_df(path)
    a+=df["Detection"].sum()
    t+=df["Total"].sum()
a=a/5
t=t/5

In [31]:
a

33658.98

In [32]:
t

37860.0

In [34]:
a

32604.79666666667

In [35]:
t

38012.6

In [37]:
32604-33658

-1054